In [1]:
from typing import  List
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_classic.schema import Document
from pydantic import BaseModel
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
!ollama pull deepseek-v2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling d8d69f2a1bfa: 100% ▕██████████████████▏ 8.9 GB                         
pulling b321cd7de6c7: 100% ▕██████████████████▏  111 B                         
pulling 4bb71764481f: 100% ▕██████████████████▏  13 KB                         
pulling 19f2fb9e8bc6: 100% ▕██████████████████▏   32 B                         
pulling 2b2c35718a16: 100% ▕██████████████████▏  494 B                         
verifying sha256 digest 
writing manifest ⠋ pulling manifest 
pulling d8d69f2a1bfa: 100% ▕██████████████████▏ 8.9 GB                         
pulling b321cd7de6c7: 100% ▕██████████████████▏  111 B                         
pulling 4bb71764481f: 100% ▕██████████████████▏  13 KB                     

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["OLLAMA_API_KEY"]=os.getenv("OLLAMA_API_KEY")
print(f"Api key is all set")

Api key is all set OLLAMA_API_KEY


In [4]:
llm=init_chat_model(
    "deepseek-v2",
    model_provider="ollama",
    temperature=0.1
)
llm.invoke("Hello who are you?").content

" Hello! I am an AI language model designed to answer questions and provide information on a wide range of topics. I'm here to help you with any questions you may have."

In [14]:
docs=TextLoader("sample_docs.txt",encoding="utf-8").load()
chunks=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50).split_documents(docs)
embedding=HuggingFaceEmbeddings()
vs=FAISS.from_documents(chunks,embedding)
retriever=vs.as_retriever()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 564.59it/s]


In [16]:
print(chunks[1])

page_content='Unlike traditional AI systems that generate a response once and stop, an AI agent follows an iterative process. It analyzes the given task, determines the next appropriate action, executes that action, observes the outcome, and uses the resulting information to decide what to do next.

This feedback mechanism allows agents to handle complex, multi-step tasks, adapt to unexpected situations, correct errors, and work toward achieving a specific goal without requiring constant human intervention.' metadata={'source': 'sample_docs.txt'}


In [26]:
## Define Agent State
class IterativeRAGState(BaseModel):
    question: str
    refined_question: str = ""
    retrieved_docs: List[Document] = []
    answer: str = ""
    verified: bool = False
    attempts: int = 0

In [27]:
#Retriever Node
def retrieve_docs(state: IterativeRAGState) -> IterativeRAGState:
    query = state.refined_question or state.question
    docs = retriever.invoke(query)
    return state.model_copy(update={"retrieved_docs": docs})

In [28]:
## Relect and verify node
def generate_answer(state: IterativeRAGState) -> IterativeRAGState:
    
    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)
    prompt = f"""Use the following context to answer the question:

Context:
{context}

Question:
{state.question}
"""
    response = llm.invoke(prompt.strip()).content.strip()
    return state.model_copy(update={"answer": response, "attempts": state.attempts + 1})

In [29]:
## Reflect On Answer
def reflect_on_answer(state: IterativeRAGState) -> IterativeRAGState:
    
    prompt = f"""
Evaluate whether the answer below is factually sufficient and complete.

Question: {state.question}
Answer: {state.answer}

Respond 'YES' if it's complete, otherwise 'NO' with feedback.
"""
    feedback = llm.invoke(prompt).content.lower()
    verified = "yes" in feedback
    return state.model_copy(update={"verified": verified})

In [30]:
## Refine Query
def refine_query(state: IterativeRAGState) -> IterativeRAGState:
    
    prompt = f"""
The answer appears incomplete. Suggest a better version of the query that would help retrieve more relevant context.

Original Question: {state.question}
Current Answer: {state.answer}
"""
    new_query = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"refined_question": new_query})


In [31]:
from langgraph.graph import StateGraph,END
builder = StateGraph(IterativeRAGState)

builder.add_node("retrieve", retrieve_docs)
builder.add_node("answer", generate_answer)
builder.add_node("reflect", reflect_on_answer)
builder.add_node("refine", refine_query)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "answer")
builder.add_edge("answer", "reflect")

builder.add_conditional_edges(
    "reflect",
    lambda s: END if s.verified or s.attempts >= 2 else "refine"
)

builder.add_edge("refine", "retrieve")
builder.add_edge("answer", END)

graph = builder.compile()

In [33]:
query = "agent loops  and transformer-based systems?"

initial_state = IterativeRAGState(question=query)
final = graph.invoke(initial_state)

print("✅ Final Answer:\n", final["answer"])
print("\n🧠 Verified:", final["verified"])
print("🔁 Attempts:", final["attempts"])


✅ Final Answer:
 The Agent Loop Architecture is a fundamental concept in the design and operation of AI agents, particularly those designed to function autonomously and iteratively. This architecture is crucial for AI agents to make decisions, interact with external tools, and improve their actions based on the results they receive. The loop can be represented as a cycle of Thinking, Acting, Observing, and Evaluating, which is repeated until the goal is achieved, a stopping condition is reached, or a maximum iteration limit is met.

The loop is structured as follows:

1. THINK: The AI agent formulates a plan or a decision based on its current understanding and the task at hand.
2. ACT: The AI agent executes the plan by interacting with external tools or resources.
3. OBSERVE: The AI agent observes the outcome of its action, which can be in the form of data or results.
4. EVALUATE: The AI agent assesses whether the goal has been achieved based on the outcome of the action.
5. REPEAT: If